# A1.12 · Cascading hallucination

**Function A — Securing AI Architectures → CyberTravels' Architecture, and Every Risk It Carries**  ·  *Security of AI*

Builds on **[A1.11 · Rogue agents in a multi-agent system](https://spbreed.github.io/cyber-commons/lessons/A1.11.html)**.

| | |
|---|---|
| Tools used | Inspect, GLM-4.6, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Let one fabricated fact travel three hops and watch its confidence rise as its provenance disappears.

**Why a security engineer needs it.** A single fabrication becomes a shared premise, and by the third hop nothing in the system records that it was ever uncertain. The control it builds is: verification against ground truth before a claim propagates (A3.5).

This is a **risk** lesson: it shows the failure happening before anything tries to stop it, so the control that follows is answering something you have already watched go wrong.

## 1 · The hook

Agent one is 90% accurate, which sounds fine. Agent two consumes its output as fact, and agent three consumes that. By the third hop the confident wrong answer has been repeated enough times that it reads like corroboration.

> **At CyberTravels.** The advisor is confident about a hotel that closed in 2024. The workflow agent books it, the file system agent validates an invoice against it, and the report to the executive cites three agreeing sources. R2.

## 2 · The framework

```
   agent 1        agent 2        agent 3        report
   90% right ---> takes as ----> takes as ----> "three sources agree"
                  fact           fact

   error compounds: 0.9 -> 0.81 -> 0.73
   confidence compounds the other way, because repetition reads as corroboration
```

**OWASP T5 — Cascading Hallucination Attacks. LLM09 — Misinformation.**

The **model** component produces confident text. Sometimes the text is wrong.
That is a known property, and on its own it is a quality problem rather than a
security one.

It becomes a security problem when the architecture has more than one step,
because an unverified claim from step one is an *input* to step two. And inputs
are not re-examined — that is the point of a pipeline.

Watch what happens to a single fabrication as it travels:

**Hop one.** "I could not find a CVE for this dependency, it is probably fine."
Hedged, and the hedge is visible.

**Hop two.** The next agent summarises: "dependency has no known CVEs."
The hedge is gone. Nothing lied — summarising removes qualifiers, that is what
summarising is.

**Hop three.** "Dependency verified clean." Now it is a finding, with the
confidence of something that was checked, and no field anywhere records that
nobody checked anything.

The security consequence is that **confidence rises as evidence disappears**,
which is exactly backwards. And it is not limited to accidents: an attacker who
can inject one plausible claim early gets it laundered into an established fact
by your own pipeline, which is why this sits in the threat taxonomy rather than
in a quality backlog.

> **Where this lands on the reference architecture.**
>
> ```
> ingress -> orchestrator -> agent_runtime -> model
>                                |              |
>                          messaging        tools / mcp
>                                |              |
>                       knowledge / memory   egress
>            identity + policy wrap every call · observability records it
> ```

## 3 · The risk, realised

One hedged guess, three hops, and the confidence it acquires on the way.

## 4 · The check, as a skill

A hedged sentence about libfoo becomes a confident claim in three hops. The skill tracks both series — confidence and surviving provenance — because either one alone looks ordinary and the pair is the finding.

In [ ]:
# skills/threats/confidence-provenance-decay-check/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: confidence-provenance-decay-check
description: >-
  Track a hedged claim across summarisation hops and measure confidence rising
  while provenance disappears. Use when output from one model or agent becomes
  input to another, in report chains, research pipelines, or any place a summary
  is summarised.
allowed-tools: Read, Grep, Glob
---

# Confidence rises at exactly the rate evidence disappears

"I could not find a CVE, it is probably fine" becomes "libfoo is clean" in
three hops. Nothing in the chain lied. Each step did what summarising does —
dropped the hedge, dropped the caveat, dropped the sentence saying where the
claim came from — and the result is a confident statement with no evidence
behind it.

## When to use this

Report generation, triage pipelines, research chains, any agent that consumes
another agent's output, and any workflow where a human reads only the last
artefact.

## Procedure

**1 — Find the chains.** Every place where generated text is input to a later
generation step. Include the human-visible summary at the end; it is a hop.

**2 — Instrument the original claim.** Record its hedge, its confidence if one
is stated, and its provenance — the source it rests on.

**3 — Step the chain and record both series.** After each hop, the claim's
confidence and its surviving provenance fields. Two numbers per hop; the shape
is the finding.

**4 — Identify the hop where provenance empties.** That is where the claim
became unfalsifiable, and it is almost always earlier than where the confidence
peaked.

**5 — Test the carry rule.** A chain that propagates confidence and provenance
as structured fields, rather than as prose the next step must re-read, does not
decay this way. Check whether the interface has those fields at all.

## Output contract

```json
{
  "chain": [{"hop": 0, "claim": "str", "confidence": 0.0, "provenance": ["str"]}],
  "confidence_delta": 0.0,
  "provenance_lost_at": 0,
  "interface": {"carries_confidence": false, "carries_provenance": false},
  "human_reads_hop": 0
}
```

## Failure modes

- **Judging the final claim on its own.** It reads well. That is the problem.
- **Measuring only confidence.** The pair is the finding; either alone is
  ordinary.
- **Fixing it with a prompt.** "Preserve caveats" is advisory; a field is not.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/threats/confidence-provenance-decay-check/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
# skills/threats/confidence-provenance-decay-check/scripts/confidence_provenance_decay_check.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Track how a hedged claim's confidence and provenance move in opposite directions across summarisation hops.

This is the executable half of the `confidence-provenance-decay-check` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

def summarise(claim, confidence):
    """Each hop compresses. Compression removes qualifiers first - they are the
    least information-dense part of a sentence."""
    for hedge in ("I could not find", "probably", "appears to", "it seems"):
        if hedge in claim:
            claim = " ".join(claim.replace(hedge, "").split())
            confidence = min(1.0, confidence + 0.3)     # certainty is what survives
    return claim.strip(", "), round(confidence, 2)

ORIGINAL = "I could not find a CVE for libfoo, it is probably fine"
claim, conf = ORIGINAL, 0.2
provenance = ["model guess, unverified"]

print(f"{'hop':>4}  {'confidence':>11}  claim")
print(f"{0:>4}  {conf:>11.2f}  {claim}")
for hop in (1, 2, 3):
    claim, conf = summarise(claim, conf)
    if hop >= 2:
        provenance = []                       # the source field is not carried on
    print(f"{hop:>4}  {conf:>11.2f}  {claim}")

print(f"\nprovenance recorded at hop 3: {provenance or 'none'}")
print(f"confidence at hop 0: 0.20   at hop 3: {conf}")
print()
print("Nothing lied. Every hop did its job. The claim gained certainty at the")
print("exact rate it lost evidence, and by hop three it reads like a finding")
print("someone verified.")
print()
print("An attacker who lands one plausible claim early gets it laundered into")
print("an established fact by your own pipeline - for free.")
assert conf >= 0.8 and not provenance

## What you just proved

A hedged guess at confidence 0.2 becomes a confident claim above 0.8 in three hops, while the provenance field empties — confidence rising at exactly the rate evidence disappears.

## Your turn

Take a finding your pipeline produced and try to walk it back to the step that first asserted it. If you cannot reach a step that checked something, you have found a cascade rather than a finding.

---

**Next → [A1.13 · Resource overload](https://spbreed.github.io/cyber-commons/lessons/A1.13.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.12.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.12.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*